# 03: WOE / IV binning (corrected version)

Changes from the previous version:
1. **Train/test split happens first.** Bins, WOE values and IV are fitted on the training vintages only, so no test-period outcome influences the scorecard.
2. **`term_months` is treated as a category** (36 or 60 months). Quantile binning collapsed it into one bin and gave it IV = 0, which wrongly removed it.
3. **Relative paths**: run this notebook from the `Python/` folder.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Run from the Python/ folder (or the repo root); paths are relative to the repo root.
ROOT = Path.cwd().parent if Path.cwd().name == 'Python' else Path.cwd()
DATA, DOCS = ROOT / 'Dataset', ROOT / 'Documentation'
DOCS.mkdir(exist_ok=True)

df = pd.read_csv(DATA / 'loans_model_ready.csv')
print(df.shape)

TARGET = 'defaulted'

# term_months has only two values (36 / 60), so it is a category, not a continuous feature.
df['term_months'] = df['term_months'].astype(int).astype(str) + ' months'

# Candidate features: deliberately excludes int_rate / grade / sub_grade (leakage, see notebook 02)
CONTINUOUS_FEATURES = [
    'loan_amnt', 'monthly_income', 'dti', 'bureau_score',
    'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'revol_util', 'total_acc'
]
CATEGORICAL_FEATURES = ['purpose', 'emp_length', 'home_ownership', 'term_months']
# addr_state excluded: 50 categories is too sparse for clean WOE bins without grouping states.

# Out-of-time split: first 70% of origination months = train, the rest = test.
vintages_sorted = sorted(df['vintage_month'].unique())
cutoff_idx = int(len(vintages_sorted) * 0.7)
train_vintages = set(vintages_sorted[:cutoff_idx])
df['split'] = np.where(df['vintage_month'].isin(train_vintages), 'train', 'test')
train = df[df['split'] == 'train'].copy()

print(f"Train: {len(train):,} loans ({min(train_vintages)} to {max(train_vintages)})")
print(f"Test:  {(df['split'] == 'test').sum():,} loans (WOE is NOT fitted on these)")

In [ ]:
def woe_iv_continuous(data_in, feature, target=TARGET, n_bins=10):
    """Quantile-bin a continuous feature and compute WOE/IV per bin."""
    data = data_in[[feature, target]].dropna().copy()
    data['bin'], bin_edges = pd.qcut(data[feature], q=n_bins, duplicates='drop', retbins=True)
    summary = _woe_table(data, 'bin', target)
    summary['feature'] = feature
    return summary, bin_edges.tolist()

def woe_iv_categorical(data_in, feature, target=TARGET):
    """Compute WOE/IV per category (each category = its own bin)."""
    data = data_in[[feature, target]].fillna('Unknown').copy()
    summary = _woe_table(data, feature, target)
    summary['feature'] = feature
    return summary, sorted(data[feature].unique().tolist())

def _woe_table(data, bin_col, target):
    grp = data.groupby(bin_col, observed=True)[target].agg(['count', 'sum'])
    grp.columns = ['total', 'bad']
    grp['good'] = grp['total'] - grp['bad']
    total_good, total_bad = grp['good'].sum(), grp['bad'].sum()
    eps = 0.5   # avoids divide-by-zero / log(0) for empty bins
    grp['dist_good'] = (grp['good'] + eps) / (total_good + eps * len(grp))
    grp['dist_bad'] = (grp['bad'] + eps) / (total_bad + eps * len(grp))
    grp['woe'] = np.log(grp['dist_good'] / grp['dist_bad'])
    grp['iv_component'] = (grp['dist_good'] - grp['dist_bad']) * grp['woe']
    grp['bad_rate'] = grp['bad'] / grp['total']
    return grp.reset_index().rename(columns={bin_col: 'bin'})

In [ ]:
woe_tables, bin_definitions, iv_summary = {}, {}, []

# Every table below is fitted on TRAIN only.
for feat in CONTINUOUS_FEATURES:
    table, edges = woe_iv_continuous(train, feat)
    woe_tables[feat] = table
    bin_definitions[feat] = {'type': 'continuous', 'edges': edges}
    iv_summary.append({'feature': feat, 'iv': table['iv_component'].sum(), 'n_bins': len(table)})

for feat in CATEGORICAL_FEATURES:
    table, categories = woe_iv_categorical(train, feat)
    woe_tables[feat] = table
    bin_definitions[feat] = {'type': 'categorical', 'categories': categories}
    iv_summary.append({'feature': feat, 'iv': table['iv_component'].sum(), 'n_bins': len(table)})

iv_df = pd.DataFrame(iv_summary).sort_values('iv', ascending=False).reset_index(drop=True)
iv_df['strength'] = pd.cut(
    iv_df['iv'], bins=[-1, 0.02, 0.1, 0.3, 0.5, np.inf],
    labels=['Not useful', 'Weak', 'Medium', 'Strong', 'Suspiciously strong: check leakage'])
iv_df

In [ ]:
top_features = iv_df.head(4)['feature'].tolist()
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, feat in zip(axes.flat, top_features):
    table = woe_tables[feat].copy()
    table['bin_label'] = table['bin'].astype(str)
    ax.bar(table['bin_label'], table['woe'], color='steelblue')
    ax.set_title(f"{feat}  (IV={iv_df.loc[iv_df['feature']==feat, 'iv'].values[0]:.3f})")
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.axhline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

In [ ]:
IV_THRESHOLD = 0.02
selected_features = iv_df[iv_df['iv'] >= IV_THRESHOLD]['feature'].tolist()
print('Features kept for modeling:', selected_features)
print('Features dropped (IV < 0.02):', [f for f in iv_df['feature'] if f not in selected_features])

In [ ]:
def apply_woe_continuous(series, edges, woe_table):
    binned = pd.cut(series, bins=edges, include_lowest=True)
    woe_map = dict(zip(woe_table['bin'].astype(str), woe_table['woe']))
    return binned.astype(str).map(woe_map)

def apply_woe_categorical(series, woe_table):
    woe_map = dict(zip(woe_table['bin'], woe_table['woe']))
    return series.fillna('Unknown').map(woe_map)

woe_df = df[['id', TARGET, 'split']].copy()
for feat in selected_features:
    if bin_definitions[feat]['type'] == 'continuous':
        woe_df[f'{feat}_woe'] = apply_woe_continuous(df[feat], bin_definitions[feat]['edges'], woe_tables[feat])
    else:
        woe_df[f'{feat}_woe'] = apply_woe_categorical(df[feat], woe_tables[feat])

# Rows outside the TRAIN bin edges (or with a category not seen in train) get the TRAIN average WOE.
is_train = (woe_df['split'] == 'train').values
for feat in selected_features:
    col = f'{feat}_woe'
    woe_df[col] = woe_df[col].fillna(woe_df.loc[is_train, col].mean())

print(woe_df.shape)
woe_df.head()

In [ ]:
woe_df.to_csv(DATA / 'loans_woe_transformed.csv', index=False)

woe_artifact = {
    'fitted_on': 'training vintages only',
    'train_vintages': [min(train_vintages), max(train_vintages)],
    'iv_summary': iv_df.to_dict(orient='records'),
    'selected_features': selected_features,
    'bin_definitions': bin_definitions,
    'woe_lookups': {
        feat: woe_tables[feat][['bin', 'woe']].astype(str).to_dict(orient='records')
        for feat in selected_features
    }
}
with open(DOCS / 'woe_scorecard_artifact.json', 'w') as f:
    json.dump(woe_artifact, f, indent=2, default=str)

print('Saved', (DATA / 'loans_woe_transformed.csv').relative_to(ROOT), woe_df.shape)
print('Saved', (DOCS / 'woe_scorecard_artifact.json').relative_to(ROOT))